# 6. Fusion, and why routing does not work

Reproduces **Section 4.5**:

| output | paper |
|---|---|
| each component alone and its error correlation with the volumetric features | `tab:errcorr` |
| grading AUC as each embedding is appended, with a paired bootstrap | `tab:matrix` |
| the per-case routing alternative | Section 4.5, the negative result |

Runs from `data_cache/` alone, in a few minutes.

## The two ways of combining

**Concatenation.** The reduced embedding is appended to the seventeen volumetric features to form
one vector per case, which the classifier then grades. No per-case weighting, no selection.

**Routing.** Each case goes to MedGemma's own grade when the model is confident and to the
volumetric classifier otherwise, with MedGemma's softmax over its grade token as the gate.

The paper reports concatenation and reports that routing failed. This notebook shows both, and
shows the diagnostic that explains why the failure is structural rather than a matter of tuning
the gate.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

from src import config
print(config.describe())

In [ ]:
from src import cohort, routing, stats, tables

blocks, y, cases = cohort.load_blocks()
cohort.inject_encoder_blocks(blocks, cases, keys=("medgemma_fewshot", "biomedclip_enh"))
component = tables.component_scores(blocks, y)

## tab:errcorr

Each component method on its own, its gap to the volumetric baseline, and how far its per-case
errors depart from the volumetric ones. The correlation is measured **within each grade**, so
agreement driven by the grade itself does not inflate it; a low value means the two methods fail
on different cases.

Expected:

| Method | AUC alone | $\Delta$ vs. Vol. | Err. corr. vs. Vol. |
|---|---|---|---|
| Volumetric features | 0.930 | --- | --- |
| MedGemma | 0.748 | -0.182 | 0.234 |
| MedGemma (few-shot) | 0.819 | -0.111 | **0.199** |
| BiomedCLIP (T1ce) | 0.830 | -0.100 | 0.365 |
| BiomedCLIP (enhancement) | 0.869 | -0.061 | 0.416 |

No embedding matches the volumetric baseline alone, and yet the MedGemma few-shot errors are the
least correlated with it. That ordering is what predicts the fusion result below: it is not the
strongest embedding that helps, it is the one that fails on different cases.

In [ ]:
errcorr = tables.table_errcorr(blocks, y, component)
display(errcorr)

published = {"Volumetric features": 0.930, "MedGemma": 0.748, "MedGemma (few-shot)": 0.819,
             "BiomedCLIP (T1ce)": 0.830, "BiomedCLIP (enhancement)": 0.869}
display(tables.compare_to_paper(errcorr, "AUC alone", published))

### The full pairwise picture

Three ways of asking the same question. The paper quotes the middle one, and also quotes from it
that the MedGemma few-shot and BiomedCLIP enhancement errors correlate 0.394 with each other.

In [ ]:
matrices = tables.errcorr_matrices(blocks, y, component)
for name, matrix in matrices.items():
    print(f"--- {name} ---")
    display(matrix)

print("few-shot against BiomedCLIP enhancement, within grade:",
      matrices["within grade"].loc["MedGemma (few-shot)", "BiomedCLIP (enhancement)"])

## tab:matrix

Grading AUC as the components of each embedding are appended to the seventeen volumetric features.
The AUC is pooled out-of-fold with a random forest, and each difference is resampled against the
volumetric baseline over cases, 2000 times.

Expected:

| Feature set | AUC | $\Delta$ | 95% CI |
|---|---|---|---|
| Volumetric features | 0.930 | --- | --- |
| + MedGemma | 0.932 | +0.002 | [-0.007, +0.011] |
| + MedGemma (few-shot) | **0.940** | +0.010 | [-0.001, +0.024] |
| + BiomedCLIP (T1ce) | 0.932 | +0.002 | [-0.008, +0.012] |
| + BiomedCLIP (enhancement) | 0.930 | +0.000 | [-0.011, +0.010] |
| + MedGemma + BiomedCLIP (T1ce) | 0.928 | -0.002 | [-0.015, +0.010] |
| + MedGemma (few-shot) + BiomedCLIP (enhancement) | 0.937 | +0.007 | [-0.008, +0.024] |

The one row that moves is the few-shot embedding, from 0.930 to 0.940, and its interval still
reaches zero. The paper reports it as a small gain with the interval attached and does not claim
significance. BiomedCLIP gains at most about +0.002, and keeping its components alongside the
few-shot ones lowers the result slightly rather than adding to it.

In [ ]:
matrix = tables.table_matrix(blocks, y)
display(matrix)

published = {"Volumetric features": 0.930, "$+$ MedGemma": 0.932,
             "$+$ MedGemma (few-shot)": 0.940, "$+$ BiomedCLIP (T1ce)": 0.932,
             "$+$ BiomedCLIP (enhancement)": 0.930,
             "$+$ MedGemma $+$ BiomedCLIP (T1ce)": 0.928,
             "$+$ MedGemma (few-shot) $+$ BiomedCLIP (enhancement)": 0.937}
display(tables.compare_to_paper(matrix, "AUC", published))

In [ ]:
display(tables.table_matrix(blocks, y, detail=True))

At four decimals the borderline nature is plain: the few-shot difference is +0.0093 with a
0.961 probability of being positive, short of a conventional bar, which is why the paper phrases it
as a small gain and prints the interval.

## The routing alternative

Concatenation lets the classifier weigh the two views globally. Routing would decide per case. The
gate is MedGemma's own confidence in the grade it wrote: the saved answer is teacher-forced up to
the assessment field and the next-token distribution is read, so this is the probability the model
actually decided with rather than a classifier fitted on top of its hidden state.

In [ ]:
p_native = routing.load_native_confidence(config.MEDGEMMA_NATIVE_CONF, cases)
p_volumetric = component["Volumetric features"]
p_medgemma = component["MedGemma"]
threshold_volumetric = stats.youden_threshold(y, p_volumetric)
threshold_medgemma = stats.youden_threshold(y, p_medgemma)

display(routing.standalone_table(y, {
    "MedGemma, its own grade token": (p_native, 0.5),
    "MedGemma, a forest over its hidden state": (p_medgemma, threshold_medgemma),
    "Volumetric features": (p_volumetric, threshold_volumetric),
}))

In [ ]:
display(routing.saturation(p_native))

MedGemma's own grade is close to chance and its confidence is heavily peaked. A saturated signal
carries little information to gate on, which is the first hint that no threshold on it will work.

### The router itself

A growing share of cases is handed to the embedding, most-confident-MedGemma first. The baseline
sits at 0% deferral, so any row that helps has to beat it.

In [ ]:
sweep, blended = routing.deferral_sweep(
    y, p_volumetric, threshold_volumetric, p_medgemma, threshold_medgemma,
    gate=routing.confidence(p_native))
display(sweep)
print(f"volumetric baseline AUC: {sweep.loc['0%', 'AUC']}")

In [ ]:
display(routing.best_router_bootstrap(
    y, p_volumetric, sweep, blended, "native-confidence gate to the embedding"))

Bootstrapping the best point of the sweep is deliberately generous: the fraction was picked after
seeing the results, so this is the most favourable test available. It still does not win.

### Why nothing routes

The decisive test. Restrict attention to the cases where the two views disagree, since those are
the only ones a router can change, and ask whether any label-free signal separates the cases where
deferring **rescues** the volumetric classifier from those where it **breaks** it. A signal that
could drive a router scores far from 0.5.

In [ ]:
flip, summary = routing.flip_separation(
    y, p_volumetric, threshold_volumetric, p_medgemma, threshold_medgemma,
    signals={
        "embedding confidence": routing.confidence(p_medgemma, threshold_medgemma),
        "disagreement between the two views": np.abs(p_volumetric - p_medgemma),
        "MedGemma's own confidence": routing.confidence(p_native),
        "MedGemma's own P(low-grade)": p_native,
        "volumetric P(low-grade), circular": p_volumetric,
    })
print(summary)
display(flip)

Every label-free signal sits near chance. The one that scores well is circular and unusable:
rescuable and breakable are *defined* by whether the volumetric classifier is wrong, which its own
probability sets, and routing on it collapses onto the majority grade -- it buys a handful of raw
correct answers by giving up minority-grade ranking, which is precisely why AUC and balanced
accuracy are the reported metrics and raw accuracy is not.

So the conclusion the paper draws is the sharper one. It is not that the embeddings repeat the
volumetric mistakes: `tab:errcorr` shows they do not, and MedGemma and the volumetric features
genuinely fail on different cases. It is that the complementarity cannot be turned into a routing
rule, because no confidence signal separates the cases MedGemma grades correctly from the ones it
does not. Concatenation is therefore the fusion reported.